# Wall & Floor Segmentation for Interior Photographs
**Nirwana AI · AI Engineer take-home**  ·  [Open in Colab](https://colab.research.google.com/github/shlokdhanokar/nirwana-wall-floor-segmentation/blob/main/notebooks/Wall_Floor_Segmentation.ipynb)

Given an interior photo, produce pixel-level **wall** and **floor** masks.

**Runtime → Run all** works on a fresh Colab runtime (CPU is enough; GPU is used if present).
Jump to **§3** to upload your own image.

**TL;DR**
* **Formulation:** 3-class dense labelling *(other / wall / floor)* rather than two independent binary problems. Wall and floor are mutually exclusive, and the hard negatives (ceiling, windows, doors, furniture) need a class of their own.
* **Model:** *WallFloorNet*, a 3.3 M-parameter encoder-decoder (MobileNetV3 encoder plus my own global-context FPN decoder), **trained by me** on ~9.2k interior scenes filtered from ADE20K. No pre-trained or fine-tuned *segmentation* model is used anywhere (see §0).
* **Classical post-processing:** a fast guided filter snaps the low-resolution predictions to real image edges at full resolution, and connected-component cleanup removes specks.
* **Baseline / first experiment:** a training-free geometric pipeline (LSD vanishing-point horizon plus GrabCut), kept for comparison. It handles floors reasonably and walls poorly. That result is the reason a learned model was needed.
* **Speed:** ONNX Runtime plus INT8 static quantisation gives a fast CPU path for a full-resolution 3558×2160 photo (numbers in §6).




## 0 · Constraint compliance & disclosures
| Constraint | How it is met |
|---|---|
| No pre-trained / fine-tuned **segmentation** model | No segmentation weights of any kind are loaded. The full segmentation network (decoder, head, loss, training loop) is defined in `wallfloor/model.py` and `wallfloor/train.py` and was trained from scratch for this task. |
| Segmentation logic developed in this solution | The label space and ADE20K remapping, the interior-scene filter, the architecture, augmentations, losses, guided-filter refinement, cleanup and the classical baseline are all written for this project. |
| No segmentation API / cloud service | Everything runs locally in this notebook (PyTorch or ONNX Runtime on CPU/GPU). |
| Disclose external models / data / libraries / AI tools | **Encoder initialisation:** torchvision's MobileNetV3-Large **ImageNet-1k classification** weights (an image classifier, not a segmenter). `--no-pretrained-encoder` trains it from random init instead. **Dataset:** ADE20K SceneParsing (MIT CSAIL), used for its *wall*, *floor* and *rug* labels only. **Libraries:** PyTorch, torchvision, OpenCV, NumPy, ONNX / ONNX Runtime, Matplotlib. **AI tools:** Claude Code (Anthropic) was used as a coding assistant. |


In [ ]:
# --- setup: get the code + trained weights (works on a fresh Colab runtime and locally) ---
import os, sys, subprocess, importlib
REPO_URL = "https://github.com/shlokdhanokar/nirwana-wall-floor-segmentation"
if os.path.isdir("wallfloor"):
    ROOT = os.getcwd()
elif os.path.isdir("../wallfloor"):
    ROOT = os.path.abspath("..")
else:
    if not os.path.isdir("repo"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
    ROOT = os.path.abspath("repo")
os.chdir(ROOT); sys.path.insert(0, ROOT)
if importlib.util.find_spec("onnxruntime") is None:   # the only dependency Colab lacks
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"], check=True)

import cv2, numpy as np, torch, matplotlib.pyplot as plt, json, time
from wallfloor.infer import WallFloorSegmenter, load_rgb
from wallfloor.classical import segment_classical
from wallfloor.metrics import binary_iou, boundary_f1
from wallfloor.viz import show, overlay, save_outputs
print("root:", ROOT, "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print(sorted(os.listdir("weights")))

## 1 · Problem formulation
* **Input:** one RGB interior photo of any size and aspect ratio. **Output:** two binary masks at the *input* resolution.
* **Label definitions** (the main ambiguity in the task):
  * *Wall*: vertical room surfaces, including paint, wallpaper, tile and stone cladding. Windows, doors, curtains, pictures and furniture in front of the wall are **not** wall.
  * *Floor*: walkable ground surface. A **rug or carpet counts as floor**, matching the reference GT supplied with the task. Furniture legs and objects on the floor are **not** floor.
  * Everything else (ceiling, windows, furniture, people, plants, ...) is *other*.
* **Why 3 classes and not 2 binary problems:** the masks must not overlap, and the softmax forces the network to explain the confusers (ceiling vs. wall, glossy floor vs. reflection, window vs. bright wall) explicitly.
* **Why learning:** wall and floor are defined by **3-D layout and semantics**, not appearance. A white wall, white ceiling and white floor can share the same colour, and a striped rug can look like wooden planks. Appearance-only rules break on that (see §5, experiment 1), so the system has to learn layout priors from data.


## 2 · Pipeline
```
image (any size) ─► resize (short side 384, /32) ─► WallFloorNet ─► softmax probs (≈384 px)
                                                      ▲  flip-TTA (optional)
                                                      │
 full-res guide (gray) ─────────────────────────────► fast guided filter ─► argmax ─► CC cleanup ─► wall, floor masks
```
**WallFloorNet** (`wallfloor/model.py`)
* *Encoder*: MobileNetV3-Large feature extractor, taps at strides 4/8/16/32. Chosen for its accuracy-per-FLOP and because it quantises well to INT8 for the CPU path.
* *Context block* (stride 32): wall vs. floor is mostly a **global layout** decision. A pixel of beige texture is floor at the bottom of the room and wall behind the sofa. The block gates a local projection with a global-average-pooled descriptor (LR-ASPP style) and concatenates an explicit global context vector.
* *Top-down FPN*: fuses stride 16 → 8 → 4 with lateral 1×1 convs and depthwise-separable 3×3 smoothing, which keeps boundaries (skirting lines, wall corners) sharp at low cost.
* *Loss*: cross-entropy (`ignore_index` for unannotated pixels, mild class weights) **plus soft Dice** on wall and floor, so large easy regions don't dominate.
* *Training data*: ADE20K filtered to **interior scenes** with a label-statistics heuristic (`wallfloor/labels.py::is_interior`: no sky, some wall/floor, ceiling or floor present). That leaves 9.2k train and 922 val images. Labels are remapped `wall→1, floor|rug→2, other→0, unlabelled→ignore`.
* *Augmentation*: scale 0.6–1.6, crops, ±8° rotation (handheld photos), **horizontal flip only** (a vertical flip would break the floor-is-down prior the model should learn), strong colour, exposure, blur and noise jitter (walls come in every colour, and interiors range from tungsten-dim to blown-out daylight).


## 3 · Run on an image
The reference image shipped with the assignment, scored against its ground-truth masks:

In [ ]:
seg = WallFloorSegmenter("weights/wallfloor_mbv3.pt", short_side=384, tta=True, refine=True)

img = load_rgb("samples/Contemporary_Stone_Accent_Lounge.webp")
out = seg(img)
show(img, out, title=f"{img.shape[1]}x{img.shape[0]}  ·  " +
     "  ".join(f"{k} {v*1000:.0f} ms" for k, v in out["timings"].items()))

gt = {k: cv2.imread(f"samples/gt/Contemporary_Stone_Accent_Lounge_{k}.png", 0) > 127 for k in ("wall", "floor")}
for k in ("wall", "floor"):
    print(f"{k:5s}  IoU = {binary_iou(out[k], gt[k]):.3f}   boundary-F = {boundary_f1(out[k], gt[k]):.3f}")
save_outputs(img, out, "results/samples", "Contemporary_Stone_Accent_Lounge")   # full-res masks as PNG

**Reading the score on this image:** the reference GT labels only the **stone accent wall** as wall. The **dark vertical slat panelling** at both sides is left out of both masks. Under the ADE20K convention the model learned, wall cladding is wall, so the model also marks the slat panels. Most of the remaining wall "error" is that **annotation-convention difference**, not a localisation error. The floor (wood planks + rug, excluding furniture legs) is where the definitions agree.

### Try your own image
On Colab, run the next cell and pick a file (or set `IMAGE_URL`). Any interior photo works, at any resolution.

In [ ]:
IMAGE_URL = ""   # optionally paste a direct image URL here
paths = []
try:
    from google.colab import files            # Colab upload widget
    up = files.upload()
    paths = list(up.keys())
except Exception:
    pass
if IMAGE_URL:
    import urllib.request
    urllib.request.urlretrieve(IMAGE_URL, "url_image.jpg"); paths.append("url_image.jpg")
if not paths:
    print("No upload - using the bundled sample image.")
    paths = ["samples/Contemporary_Stone_Accent_Lounge.webp"]
for p in paths:
    im = load_rgb(p)
    o = seg(im)
    show(im, o, title=os.path.basename(p))
    save_outputs(im, o, "results/uploads", os.path.splitext(os.path.basename(p))[0])
print("masks saved to results/uploads/")

## 4 · What each stage contributes
Network probabilities at ~384 px vs. the full-resolution result after guided-filter refinement (zoomed crops):

In [ ]:
raw = WallFloorSegmenter("weights/wallfloor_mbv3.pt", tta=False, refine=False)(img)
ref = WallFloorSegmenter("weights/wallfloor_mbv3.pt", tta=False, refine=True)(img)
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].imshow(out["probs"][..., 1], cmap="magma"); ax[0].set_title("P(wall) - network output")
ax[1].imshow(out["probs"][..., 2], cmap="magma"); ax[1].set_title("P(floor) - network output")
ax[2].imshow(out["probs"].argmax(2), cmap="viridis"); ax[2].set_title("argmax (0 other / 1 wall / 2 floor)")
[a.axis("off") for a in ax]; plt.show()

H, W = img.shape[:2]
y0, y1, x0, x1 = int(.50*H), int(.80*H), int(.30*W), int(.62*W)    # sofa / coffee-table / rug region
fig, ax = plt.subplots(1, 2, figsize=(18, 6))
for a, (o, t) in zip(ax, [(raw, "bilinear upsampling only"), (ref, "guided-filter refinement")]):
    a.imshow(overlay(img, o["wall"], o["floor"])[y0:y1, x0:x1]); a.set_title(t); a.axis("off")
plt.show()
for name, o in [("raw", raw), ("refined", ref)]:
    print(f"{name:8s} floor IoU {binary_iou(o['floor'], gt['floor']):.3f}  BF {boundary_f1(o['floor'], gt['floor']):.3f} | "
          f"wall IoU {binary_iou(o['wall'], gt['wall']):.3f}  BF {boundary_f1(o['wall'], gt['wall']):.3f}")

## 5 · Experiments & iteration
**Experiment 1: training-free geometric baseline** (`wallfloor/classical.py`). This came first, to see how far image-formation priors alone go:
1. LSD line segments → pairwise intersections of oblique segments → weighted-median **horizon** (Manhattan-world vanishing geometry).
2. **GrabCut** seeded with *floor below the horizon* / *definite floor in a bottom-centre band*, then a second GrabCut for *wall between ceiling band and horizon*.

It gets the floor of the reference image largely right but leaks wall into the bright window and curtains. That is expected: GrabCut's colour GMMs cannot tell a white wall from a white window or ceiling. Walls are defined by geometry and semantics, not by colour. It is also slow (~seconds per image) because of the iterative graph cuts. This failure is what motivated a learned model.

In [ ]:
t0 = time.perf_counter(); cl = segment_classical(img); t_cl = time.perf_counter() - t0
show(img, cl, title=f"Classical baseline (horizon at {cl['horizon']:.2f}·H) - {t_cl:.1f}s")
for k in ("wall", "floor"):
    print(f"classical {k:5s} IoU {binary_iou(cl[k], gt[k]):.3f}   vs  WallFloorNet {binary_iou(out[k], gt[k]):.3f}")

**Experiments 2–6: learned model, refinement, TTA, resolution, quantisation.** Quantitative comparison on the
held-out **ADE20K interior validation split** (922 images, never trained on) and on the reference image.
IoU is per-class over the whole split. BF is the boundary F-score at a tolerance of 0.75 % of the image diagonal, so it measures edge quality, which IoU barely reflects.
Latency is end-to-end on the 3558×2160 reference photo (CPU, 4 threads, median of 10 runs), *including* resize, refinement and cleanup.

_(run scripts/evaluate.py)_


In [ ]:
import pandas as pd
hist = pd.DataFrame(json.load(open("weights/wallfloor_mbv3_history.json")))
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
ax[0].plot(hist.epoch, hist.loss); ax[0].set_title("train loss (CE + 0.5·Dice)"); ax[0].set_xlabel("epoch")
for c in ("iou_wall", "iou_floor", "miou_wall_floor"):
    ax[1].plot(hist.epoch, hist[c], label=c)
ax[1].legend(); ax[1].set_title("ADE20K-interior val IoU"); ax[1].set_xlabel("epoch"); plt.show()
pd.DataFrame(json.load(open("results/metrics.json"))).round(3)

Held-out validation images, **worst / median / best** by per-image mIoU (prediction row, then ground-truth row). The failure cases matter as much as the successes:

In [ ]:
for tag in ("worst", "median", "best"):
    p = f"results/gallery/val_{tag}.jpg"
    if os.path.exists(p):
        plt.figure(figsize=(18, 14)); plt.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
        plt.title(tag); plt.axis("off"); plt.show()

## 6 · Performance optimisation (bonus)
Levers, in order of impact:
1. **Architecture budget:** MobileNetV3 plus a 0.34 M-parameter decoder: 3.3 M params in total, 13 MB FP32.
2. **Decouple network resolution from output resolution:** the network sees ~384 px. Full resolution comes from the guided filter, where only the per-pixel linear coefficients are upsampled, so cost is O(pixels) with a tiny constant. Running the network at full 3558×2160 would be roughly 50× more compute and *less* accurate, because the receptive field would cover a smaller share of the room.
3. **ONNX Runtime** graph optimisations (operator fusion) for the CPU path.
4. **INT8 static quantisation** (QDQ, per-channel weights, calibrated on 64 interior images): 4× smaller model, faster convolutions on CPU, with the IoU change measured in the table above.
5. Flip-TTA is optional: about +2× network time for a small accuracy gain. It is a switch for quality-vs-latency, off in the fast path.

Live benchmark on *this* runtime:

In [ ]:
def bench(s, im, n=5):
    s(im); ts = []
    for _ in range(n):
        t0 = time.perf_counter(); s(im); ts.append(time.perf_counter() - t0)
    return 1000 * float(np.median(ts))

threads = os.cpu_count()
cfgs = [("PyTorch FP32 CPU, TTA", WallFloorSegmenter("weights/wallfloor_mbv3.pt", tta=True, device="cpu", threads=threads)),
        ("PyTorch FP32 CPU",      WallFloorSegmenter("weights/wallfloor_mbv3.pt", tta=False, device="cpu", threads=threads)),
        ("ONNX FP32 CPU",         WallFloorSegmenter("weights/wallfloor_mbv3.onnx", tta=False, threads=threads)),
        ("ONNX INT8 CPU",         WallFloorSegmenter("weights/wallfloor_mbv3_int8.onnx", tta=False, threads=threads))]
if torch.cuda.is_available():
    cfgs.append(("PyTorch GPU (AMP), TTA", WallFloorSegmenter("weights/wallfloor_mbv3.pt", tta=True, device="cuda")))
rows = []
for name, s in cfgs:
    o = s(img)
    rows.append({"config": name, "latency_ms (3558x2160)": round(bench(s, img)),
                 "IoU wall": round(binary_iou(o["wall"], gt["wall"]), 3), "IoU floor": round(binary_iou(o["floor"], gt["floor"]), 3)})
for f in ("wallfloor_mbv3.pt", "wallfloor_mbv3.onnx", "wallfloor_mbv3_int8.onnx"):
    print(f"{f:28s} {os.path.getsize('weights/' + f)/1e6:5.1f} MB")
pd.DataFrame(rows)

## 7 · Reproduce training (optional, ~1 h on a Colab T4)
Set `RUN_TRAINING = True` to download ADE20K (~1 GB), rebuild the interior subset and retrain from scratch. The trained weights in `weights/` were produced by exactly this code.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    if not os.path.isdir("data/ADEChallengeData2016"):
        os.makedirs("data", exist_ok=True)
        subprocess.run(["wget", "-q", "-O", "data/ade.zip",
                        "http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip"], check=True)
        subprocess.run(["unzip", "-q", "data/ade.zip", "-d", "data"], check=True)
    subprocess.run([sys.executable, "-m", "wallfloor.prepare", "--ade-root", "data/ADEChallengeData2016",
                    "--out", "data/cache", "--workers", "2"], check=True)
    subprocess.run([sys.executable, "-m", "wallfloor.train", "--cache", "data/cache", "--out", "weights/retrained.pt",
                    "--epochs", "40", "--batch", "16", "--workers", "2"], check=True)
    subprocess.run([sys.executable, "-m", "wallfloor.export", "--weights", "weights/retrained.pt"], check=True)

## 8 · Technical discussion

### Why this approach, and what it assumes about interior images
* **Wall and floor are layout concepts, not appearance concepts.** The same beige texture is floor at the bottom of the frame and wall behind the sofa. A white wall, white ceiling and white window can be colour-identical. Any method that decides pixel-by-pixel from colour and texture is capped well below what's needed (experiment 1). The model has to reason about **where a surface sits in the room**. That is why the decoder has an explicit global-context path, and why the network runs at a moderate resolution where its receptive field covers the whole room.
* **A learned model, trained by me, under the "no pre-trained segmentation model" rule.** The useful knowledge (rooms have a floor at the bottom, walls meet it at a skirting line, rugs lie on floors, windows sit in walls) is easiest to acquire from labelled interiors. ADE20K provides wall/floor/rug labels for ~9k interiors. The segmentation network was designed and trained from scratch for this task. Only the encoder starts from generic ImageNet-classification features, and that option is disclosed and switchable.
* **Classical CV where it is strongest:** edge-aware upsampling (guided filter) and topology cleanup are cheap, deterministic and explainable, and they fix the network's main weakness: coarse boundaries.
* **Assumptions:** (1) the photo is an interior taken roughly upright (±~10° roll; the model sees ±8° rotations in training); (2) "floor" includes rugs and carpets and excludes objects standing on it; (3) "wall" includes any wall finish (paint, tile, stone, panelling) and excludes openings (windows, doors) and objects hung on it; (4) typical perspective photography. Fisheye or 360° panoramas are out of distribution.

### Hardest cases and how they were handled
| Case | Why it is hard | Handling |
|---|---|---|
| White wall vs. white ceiling vs. bright window | Same colour, no texture | Global-context block plus training data where ceilings and windows are explicit *other*; GrabCut failed exactly here |
| Floor vs. rug vs. wood-look panelling | Rugs have strong patterns; plank textures appear on walls too | Rug merged into floor in the label map (matches the provided GT); position/layout context separates floor planks from wall planks |
| Thin structures (chair and table legs, pendant cords) | Thinner than one network pixel at 384 px | Guided filter recovers edges from the full-res image; tiny holes < 0.05 % of the image are filled, larger holes (furniture) are preserved |
| Reflections on glossy floors / mirrors | Reflection of a wall *looks like* a wall | Partially handled by context; remains a failure mode (see gallery) |
| Annotation-convention ambiguity (e.g. slat panelling, built-in cabinets, wardrobes) | Graders and datasets disagree on what "wall" is | Made explicit in §1. In a product, the convention would be fixed with the design team and encoded in the training labels |
| Arbitrary resolutions / aspect ratios | 3558×2160 input would be expensive and shift the receptive field | Network runs on short side 384 (aspect preserved, /32), output upsampled with guided filter |

### Alternatives considered
1. **Purely classical** (vanishing points + GrabCut / superpixel region growing). Implemented as the baseline: no data needed, explainable, but fails on colour-ambiguous scenes and is slower. Kept as a fallback and as evidence for the decision.
2. **Room-layout estimation** (fit a cuboid room, e.g. LayoutNet/HorizonNet-style corner and edge regression), then subtract objects. It gives very clean planar walls, but it needs layout-annotated data, assumes a cuboid and still needs a separate "clutter" mask. A good second stage, though, see below.
3. **Monocular depth + surface normals → plane fitting** (floor = large horizontal plane at the bottom, walls = vertical planes). It would need a pre-trained depth network. Possibly allowed, since it isn't a segmentation model, but it is a grey area under the rules and a heavy dependency, so it was not used.
4. **Self-supervised features (DINOv2/CLIP) + a small trained head.** Probably the most accurate per hour of work, but a large backbone, slower on CPU, and closer to the spirit of "using a foundation segmenter" than I wanted.
5. **From-scratch encoder** (no ImageNet init). Supported by `--no-pretrained-encoder`. It converges more slowly and generalises somewhat worse from 9k images. ImageNet initialisation is the standard, disclosed trade-off.

### How mask quality was evaluated
* **Held-out quantitative:** per-class IoU and mIoU over the 922-image ADE20K interior *validation* split. It is never used for training, only for picking the best epoch.
* **Boundary F-score** (tolerance 0.75 % of the diagonal), because IoU is dominated by large interior regions and barely moves when edges are off by 10 px. Edge quality is what a re-texturing product actually shows the user.
* **Reference image** supplied with the task: IoU/BF against its GT masks, with the slat-panel convention mismatch made explicit.
* **Qualitative stratified review:** worst / median / best validation images by per-image mIoU, so failure modes are inspected rather than cherry-picked successes.
* **Every pipeline change is an ablation row** (refinement on/off, TTA, input resolution, FP32 vs INT8) with quality *and* latency, so trade-offs are explicit.

### What I would do next for real-world robustness
1. **Data:** collect and label a few thousand in-domain photos (real-estate listings, phone photos, renders like the reference image) under the *product's* wall/floor convention. Add hard-negative mining of mirrors, glass partitions, glossy floors, open-plan spaces and empty rooms. Label noise and convention mismatch currently cost more than model capacity.
2. **Geometry-aware second stage:** fit planes or a room layout to the predicted wall/floor regions, regularise each wall to a planar polygon, and enforce wall–floor adjacency at a single skirting line. That gives crisp, product-grade boundaries and lets one wall be separated from another (useful for per-wall re-texturing).
3. **Instance-level walls** (each wall plane separately), plus an explicit *ceiling* class.
4. **Larger capacity where latency allows**, e.g. a higher-resolution second pass on the boundary band only, or a heavier encoder on GPU servers with the current model as the on-device path.
5. **Uncertainty output:** expose per-pixel confidence (the softmax margin is already computed) so the product can ask the user to confirm low-confidence regions instead of failing silently.
6. **Calibration of the INT8 model** on in-domain photos, quantisation-aware training if accuracy drops, and a TensorRT / CoreML build for edge deployment.

### Limitations (honest summary)
* Learned priors come from ADE20K's annotation conventions, which differ from the reference GT on panelling and cladding.
* No explicit 3-D reasoning, so mirrors and reflections can be labelled as the surface they reflect.
* Very unusual viewpoints (top-down, strong roll, fisheye) and non-residential interiors with little visible floor are weaker.
* A single "wall" mask merges all wall planes. Separate wall instances would need the next step above.
